In [1]:
from pathlib import Path
import polars as pl

In [4]:
DATA_FOLDER_PATH = Path("D:/TradingData/massive_flatfiles/us_stock_sip/minutes_agg_v1/2024/05")
DATE_TO_PICK = "2024-05-01"
DATA_PATH = DATA_FOLDER_PATH / f"{DATE_TO_PICK}.csv.gz"
mb = pl.read_csv(DATA_PATH)
mb

ticker,volume,open,close,high,low,window_start,transactions
str,i64,f64,f64,f64,f64,i64,i64
"""A""",2351,137.04,137.04,137.04,137.04,1714568520000000000,1
"""A""",16847,136.72,136.595,136.88,136.595,1714570200000000000,51
"""A""",1235,136.13,136.785,136.86,136.13,1714570260000000000,52
"""A""",1576,137.0,137.29,137.51,137.0,1714570320000000000,46
"""A""",2926,137.0919,138.14,138.14,137.0919,1714570380000000000,62
…,…,…,…,…,…,…,…
"""ZYXI""",2418,11.73,11.75,11.75,11.73,1714593300000000000,63
"""ZYXI""",3670,11.76,11.75,11.77,11.75,1714593360000000000,51
"""ZYXI""",1579,11.76,11.74,11.76,11.74,1714593420000000000,29


## Opening Box Scanner

This section builds a parameterized opening box for each ticker, joins prior daily statistics from the same flatfile folder, computes the same setup metrics used by the QuantConnect scanner, and sorts by `final_score`.

For Polygon/Massive minute aggregates, `window_start` is the bar start time in UTC nanoseconds. A `09:30` to `09:35` box therefore uses bars with starts `09:30`, `09:31`, `09:32`, `09:33`, and `09:34` in New York time.


In [ ]:
from datetime import date, datetime

# Scanner parameters. Change these to test other dates and opening-box windows.
SCAN_DATE = DATE_TO_PICK
BOX_START_TIME = "09:30"
BOX_END_TIME = "09:35"
MARKET_TIMEZONE = "America/New_York"
LOOKBACK_DAILY_FILES = 20
DAILY_LOOKBACK_ROOT = DATA_FOLDER_PATH.parent
MIN_BOX_BARS = 1

# QuantConnect scanner parameters mirrored here.
MIN_PRICE = 5.0
MIN_AVG_DAILY_VOLUME = 1_000_000
MIN_ATR = 0.50
EXPECTED_BOX_DAILY_VOLUME_SHARE = 0.02
MIN_OPENING_RELATIVE_VOLUME = 0.75
MIN_SETUP_SCORE = 45.0
MIN_GAP_UP_PCT = 0.005
MIN_CLOSE_LOCATION = 0.60
MIN_BODY_TO_RANGE = 0.20
MIN_RANGE_ATR_FRACTION = 0.05
MAX_RANGE_ATR_FRACTION = 0.80
IDEAL_RANGE_ATR_FRACTION = 0.30
LIQUIDITY_SCORE_VOLUME = 10_000_000


def parse_hhmm(value: str) -> int:
    hour, minute = map(int, value.split(":"))
    return hour * 60 + minute


def date_from_path(path: Path) -> date:
    return datetime.strptime(path.stem.removesuffix(".csv"), "%Y-%m-%d").date()


def add_market_time_columns(frame, timezone: str):
    return frame.with_columns(
        pl.from_epoch("window_start", time_unit="ns")
        .dt.replace_time_zone("UTC")
        .dt.convert_time_zone(timezone)
        .alias("bar_time")
    ).with_columns(
        pl.col("bar_time").dt.date().alias("bar_date"),
        (
            pl.col("bar_time").dt.hour() * 60
            + pl.col("bar_time").dt.minute()
        ).alias("minute_of_day"),
    )


def build_opening_box(
    minute_bars: pl.DataFrame,
    scan_date: str,
    box_start_time: str,
    box_end_time: str,
    timezone: str,
    min_box_bars: int,
) -> pl.DataFrame:
    scan_day = datetime.strptime(scan_date, "%Y-%m-%d").date()
    start_minute = parse_hhmm(box_start_time)
    end_minute = parse_hhmm(box_end_time)

    prepared = add_market_time_columns(minute_bars, timezone)

    return (
        prepared
        .filter(
            (pl.col("bar_date") == scan_day)
            & (pl.col("minute_of_day") >= start_minute)
            & (pl.col("minute_of_day") < end_minute)
        )
        .sort(["ticker", "window_start"])
        .group_by("ticker")
        .agg(
            pl.first("open").alias("box_open"),
            pl.last("close").alias("box_close"),
            pl.max("high").alias("box_high"),
            pl.min("low").alias("box_low"),
            pl.sum("volume").alias("box_volume"),
            pl.sum("transactions").alias("box_transactions"),
            pl.count().alias("box_bar_count"),
            pl.min("bar_time").alias("box_first_bar_time"),
            pl.max("bar_time").alias("box_last_bar_time"),
        )
        .filter(pl.col("box_bar_count") >= min_box_bars)
    )


def prior_data_paths(data_folder: Path, scan_date: str, lookback_files: int) -> list[Path]:
    scan_day = datetime.strptime(scan_date, "%Y-%m-%d").date()
    dated_paths = []

    for candidate in data_folder.rglob("*.csv.gz"):
        try:
            candidate_day = date_from_path(candidate)
        except ValueError:
            continue

        if candidate_day < scan_day:
            dated_paths.append((candidate_day, candidate))

    return [path for _, path in sorted(dated_paths)[-lookback_files:]]


def load_prior_daily_stats(
    data_folder: Path,
    scan_date: str,
    timezone: str,
    lookback_files: int,
) -> pl.DataFrame:
    paths = prior_data_paths(data_folder, scan_date, lookback_files)

    if not paths:
        return pl.DataFrame(
            schema={
                "ticker": pl.String,
                "previous_close": pl.Float64,
                "avg_daily_volume_14": pl.Float64,
                "atr_14": pl.Float64,
                "daily_rows": pl.UInt32,
            }
        )

    scans = []

    for source_path in paths:
        session_date = date_from_path(source_path)
        scans.append(
            pl.scan_csv(source_path)
            .select("ticker", "volume", "open", "close", "high", "low", "window_start")
            .with_columns(pl.lit(session_date).alias("session_date"))
        )

    daily = (
        pl.concat(scans)
        .sort(["ticker", "session_date", "window_start"])
        .group_by(["ticker", "session_date"])
        .agg(
            pl.first("open").alias("daily_open"),
            pl.last("close").alias("daily_close"),
            pl.max("high").alias("daily_high"),
            pl.min("low").alias("daily_low"),
            pl.sum("volume").alias("daily_volume"),
        )
        .sort(["ticker", "session_date"])
        .with_columns(pl.col("daily_close").shift(1).over("ticker").alias("prior_daily_close"))
        .with_columns(
            pl.max_horizontal(
                pl.col("daily_high") - pl.col("daily_low"),
                (pl.col("daily_high") - pl.col("prior_daily_close")).abs(),
                (pl.col("daily_low") - pl.col("prior_daily_close")).abs(),
            ).alias("true_range")
        )
        .collect()
    )

    return (
        daily
        .group_by("ticker")
        .agg(
            pl.last("daily_close").alias("previous_close"),
            pl.col("daily_volume").tail(14).mean().alias("avg_daily_volume_14"),
            pl.col("true_range").drop_nulls().tail(14).mean().alias("atr_14"),
            pl.count().alias("daily_rows"),
        )
    )


def score_boxes(boxes: pl.DataFrame, daily_stats: pl.DataFrame) -> pl.DataFrame:
    scored = boxes.join(daily_stats, on="ticker", how="left")

    return (
        scored
        .with_columns(
            (pl.col("box_high") - pl.col("box_low")).alias("box_range"),
            ((pl.col("box_high") + pl.col("box_low")) / 2.0).alias("box_mid"),
            (pl.col("avg_daily_volume_14") * EXPECTED_BOX_DAILY_VOLUME_SHARE).alias("expected_box_volume"),
        )
        .with_columns(
            (pl.col("box_volume") / pl.col("expected_box_volume")).alias("orb_relative_volume"),
            (pl.col("box_range") / pl.col("atr_14")).alias("range_atr"),
            ((pl.col("box_close") - pl.col("box_low")) / pl.col("box_range")).alias("close_location"),
            ((pl.col("box_close") - pl.col("box_open")).abs() / pl.col("box_range")).alias("body_to_range"),
            ((pl.col("box_open") / pl.col("previous_close")) - 1.0).alias("gap_pct"),
        )
        .with_columns(
            ((1.0 - ((pl.col("range_atr") - IDEAL_RANGE_ATR_FRACTION).abs() / IDEAL_RANGE_ATR_FRACTION)).clip(0.0, 1.0)).alias("ideal_range_score"),
            (pl.col("orb_relative_volume").clip(0.0, 10.0) / 10.0).alias("rv_score"),
            (pl.col("gap_pct").clip(0.0, 0.10) / 0.10).alias("gap_score"),
            (pl.col("avg_daily_volume_14") / LIQUIDITY_SCORE_VOLUME).clip(0.0, 1.0).alias("liquidity_score"),
        )
        .with_columns(
            (
                35.0 * pl.col("rv_score")
                + 20.0 * pl.col("close_location")
                + 15.0 * pl.col("gap_score")
                + 15.0 * pl.col("ideal_range_score")
                + 15.0 * pl.col("liquidity_score")
            ).alias("final_score")
        )
        .with_columns(
            (
                (pl.col("box_close") >= MIN_PRICE)
                & (pl.col("avg_daily_volume_14") >= MIN_AVG_DAILY_VOLUME)
                & (pl.col("atr_14") >= MIN_ATR)
                & (pl.col("orb_relative_volume") >= MIN_OPENING_RELATIVE_VOLUME)
                & (pl.col("box_high") > pl.col("box_low"))
                & (pl.col("gap_pct") >= MIN_GAP_UP_PCT)
                & (pl.col("box_close") > pl.col("box_open"))
                & (pl.col("range_atr") >= MIN_RANGE_ATR_FRACTION)
                & (pl.col("range_atr") <= MAX_RANGE_ATR_FRACTION)
                & (pl.col("close_location") >= MIN_CLOSE_LOCATION)
                & (pl.col("body_to_range") >= MIN_BODY_TO_RANGE)
                & (pl.col("final_score") >= MIN_SETUP_SCORE)
            ).fill_null(False).alias("passes_qc_setup_filter")
        )
        .with_columns(pl.col("final_score").fill_null(-1.0).alias("_sort_score"))
        .sort("_sort_score", descending=True)
        .drop("_sort_score")
    )


SCAN_DATA_PATH = DATA_FOLDER_PATH / f"{SCAN_DATE}.csv.gz"
scan_minute_bars = mb if SCAN_DATE == DATE_TO_PICK else pl.read_csv(SCAN_DATA_PATH)

box_df = build_opening_box(
    scan_minute_bars,
    scan_date=SCAN_DATE,
    box_start_time=BOX_START_TIME,
    box_end_time=BOX_END_TIME,
    timezone=MARKET_TIMEZONE,
    min_box_bars=MIN_BOX_BARS,
)

daily_stats_df = load_prior_daily_stats(
    DAILY_LOOKBACK_ROOT,
    scan_date=SCAN_DATE,
    timezone=MARKET_TIMEZONE,
    lookback_files=LOOKBACK_DAILY_FILES,
)

scanner_df = score_boxes(box_df, daily_stats_df)

scanner_output_df = scanner_df.select(
    "ticker",
    "final_score",
    "passes_qc_setup_filter",
    "box_open",
    "box_high",
    "box_mid",
    "box_low",
    "box_close",
    "box_volume",
    "box_bar_count",
    "orb_relative_volume",
    "range_atr",
    "close_location",
    "body_to_range",
    "gap_pct",
    "previous_close",
    "avg_daily_volume_14",
    "atr_14",
    "daily_rows",
    "box_first_bar_time",
    "box_last_bar_time",
)

scanner_output_df
